# 独自DINOv2 headのReport不要推論
学習後のbest.ptと、学習cache作成時と同じ汎用encoder・source・provenanceをInputへ手動で追加します。単独の候補です。p003へ自動混合・代理提出はしません。

In [ ]:
from pathlib import Path
import json, sys, zipfile
CODE_INPUT = Path('/kaggle/input/your-frozen-feature-code')  # Edit code Input ONLY; never scan the Competition tree.
if not CODE_INPUT.is_dir():
    raise RuntimeError('Set CODE_INPUT to the manually attached code Input directory')
bundles = list(CODE_INPUT.rglob('rsna-frozen-features-code.zip'))
if len(bundles) > 1:
    raise RuntimeError('Multiple code ZIPs found; select one immutable code Input')
if len(bundles) == 1:
    CODE = Path('/kaggle/working/frozen-code')
    if CODE.exists():
        raise RuntimeError('Fresh kernel/output directory required')
    CODE.mkdir()
    with zipfile.ZipFile(bundles[0]) as archive:
        for name in archive.namelist():
            dest = (CODE / name).resolve()
            if not dest.is_relative_to(CODE.resolve()):
                raise ValueError('Unsafe archive member')
        archive.extractall(CODE)
else:
    roots = [p.parents[2] for p in CODE_INPUT.rglob('feature_contracts.py') if p.parent.name == 'rsna_knee' and p.parent.parent.name == 'src']
    if len(roots) != 1:
        raise RuntimeError('Attach exactly one feature code bundle, ZIP or extracted src layout')
    CODE = roots[0]
sys.path.insert(0, str(CODE / 'src'))
import torch
assert torch.cuda.is_available(), 'CUDA GPU required'
print(torch.__version__, torch.cuda.get_device_name(0))


In [ ]:
# Edit these attached Input paths. Internet OFF. No file is downloaded by this notebook.
ROOT = Path('/kaggle/input/competitions/rsna-knee-abnormality-detection')
if not ROOT.exists():
    ROOT = Path('/kaggle/input/rsna-knee-abnormality-detection')
ENCODER_REPO = Path('/kaggle/input/your-generic-dinov2/dinov2-source')
ENCODER_WEIGHTS = Path('/kaggle/input/your-generic-dinov2/dinov2_vits14_pretrain.pth')
PROVENANCE = Path('/kaggle/input/your-generic-dinov2/encoder-provenance.json')
provenance = json.loads(PROVENANCE.read_text(encoding='utf-8-sig'))


In [ ]:
from rsna_knee.feature_runtime import load_head_checkpoint, predict_feature_cache
from rsna_knee.feature_imaging import extract_cache
from rsna_knee.feature_contracts import feature_contract, json_hash
CHECKPOINT = Path('/kaggle/input/your-feature-head/best.pt')
checkpoint = load_head_checkpoint(CHECKPOINT)
CONFIG = checkpoint['config']
assert feature_contract(CONFIG, provenance) == checkpoint['feature_contract'], 'Encoder or preprocessing changed'
CACHE = Path('/kaggle/working/frozen-test-v1')
extract_cache(ROOT, 'test', CACHE, CONFIG, provenance, ENCODER_REPO, ENCODER_WEIGHTS)
result = predict_feature_cache(ROOT / 'test.csv', CACHE, CHECKPOINT, '/kaggle/working/submission.csv',
                               sample_csv=ROOT / 'sample_submission.csv')
print(result)


submission.receipt.jsonのvalid、対象UID一致、12所見順、finite値を確認します。別途保存実行が完走した後、Kaggle上で手動提出します。Visible test完走はHidden testの9時間保証ではありません。